# 유튜브 크롤링

## 3개 TEST

In [ ]:
import yt_dlp
import pandas as pd

def get_youtube_subscribers(url):
    ydl_opts = {
        'quiet': True, #로그안뜨게 조용히!
        'no_warnings': True,
        'extract_flat': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=False)
            
            channel_id = info.get('channel_id', '알 수 없음')
            channel_name = info.get('channel') or info.get('uploader', '알 수 없음')
            sub_count = info.get('channel_follower_count', None)
            
            return {
                '채널ID': channel_id,
                '채널명': channel_name,
                '구독자 수': sub_count
            }
            
        except Exception as e:
            print(f"❌ [{url}] 에러 발생: {e}")
            return None 

# 테스트할 채널 URL 리스트
urls = [
    "https://www.youtube.com/@yerang_garang",
    "https://www.youtube.com/@vege_market",
    "https://www.youtube.com/@ENHYPENOFFICIAL"
]

data_list = []

total_urls = len(urls)
print(f"총 {total_urls}개의 채널 데이터 수집 시작\n")

for index, url in enumerate(urls, start=1):
    print(f"[{index}/{total_urls}] 수집 중... ({url})") # (1/3) 형태로 출력
    
    result = get_youtube_subscribers(url)
    if result:
        data_list.append(result)

df = pd.DataFrame(data_list)

print("\n" + "="*60)
print("수집 완료")
print("="*60)
print(df)

총 3개의 채널 데이터 수집 시작

[1/3] 수집 중... (https://www.youtube.com/@yerang_garang)
[2/3] 수집 중... (https://www.youtube.com/@vege_market)
[3/3] 수집 중... (https://www.youtube.com/@ENHYPENOFFICIAL)

수집 완료
                       채널ID               채널명     구독자 수
0  UClUE7d0UQ7dhLjK1r6xpG9w              예랑가랑    379000
1  UC3brZSpeNMyfT5dcA8vPjUQ  야채상회 Vege Market     24800
2  UCArLZtok93cO5R9RI4_Y5Jw           ENHYPEN  13600000


In [8]:
display(df)

,채널ID,채널명,구독자 수
0,UClUE7d0UQ7dhLjK1r6xpG9w,예랑가랑,379000
1,UC3brZSpeNMyfT5dcA8vPjUQ,야채상회 Vege Market,24800
2,UCArLZtok93cO5R9RI4_Y5Jw,ENHYPEN,13600000


## 수임님이 주신 엑셀에서 youtube url 추출 후 구독자 수 수집

링크 겹치는 것들 있음

In [27]:
import pandas as pd

# 1. 파일 읽기 (한글 깨짐 방지를 위해 cp949 추가)
df = pd.read_csv('bj_links_extracted_full.csv')

# 2. 개수 출력 (count: 채워진 것만, nunique: 중복 제외)
print(f"1. 유튜브 링크가 채워진 개수 (널 제외): {df['youtube_url'].count()}개")
print(f"2. 고유 유튜브 링크 개수 (중복 제외): {df['youtube_url'].nunique()}개")

1. 유튜브 링크가 채워진 개수 (널 제외): 6144개
2. 고유 유튜브 링크 개수 (중복 제외): 6118개


겹치는 링크 확인

In [29]:
import pandas as pd

# 1. 파일 읽기
try:
    df = pd.read_csv('bj_links_extracted_full.csv', encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv('bj_links_extracted_full.csv', encoding='cp949')

# 2. 중복된 URL 찾기 (2번 이상 등장한 URL만 골라내기)
url_counts = df['youtube_url'].value_counts()
duplicate_urls = url_counts[url_counts > 1].index.tolist()

print(f"총 {len(duplicate_urls)}종류의 URL이 중복 사용 중입니다.\n")

# 3. 중복된 URL별로 어떤 스트리머가 있는지 출력
for url in duplicate_urls:
    streamers = df[df['youtube_url'] == url]['streamer_name'].tolist()
    
    print(f"🔗 URL: {url}")
    print(f"👥 사용자({len(streamers)}명): {', '.join(streamers)}")
    print("-" * 50)

총 22종류의 URL이 중복 사용 중입니다.

🔗 URL: https://www.youtube.com/@AfterLive_kr
👥 사용자(3명): 아마네루 피요, 콘야 유메이, 시노 레이
--------------------------------------------------
🔗 URL: https://www.youtube.com/@girlz_one_official
👥 사용자(3명): 세라SERA#, 아이링IRING, 춘리CHUNLI
--------------------------------------------------
🔗 URL: https://www.youtube.com/
👥 사용자(3명): Hebi., PLAVE 플레이브_UCPZIPuQPrfrUG9Xe, 이원향
--------------------------------------------------
🔗 URL: https://www.youtube.com/@poresees
👥 사용자(3명): 리유 Lee you, 유쿠 yuku, 클 렌 klen
--------------------------------------------------
🔗 URL: https://www.youtube.com/channel/UCSR-ifIl_RgeF1-CJTmCJ2Q
👥 사용자(2명): 뀨웅o, 뀨웅_
--------------------------------------------------
🔗 URL: https://www.youtube.com/@Lucihan_bmk
👥 사용자(2명): 루시한, 루시한데뷔계정
--------------------------------------------------
🔗 URL: https://www.youtube.com/@%EB%A7%88%EB%A5%B4%EB%B6%80
👥 사용자(2명): 마르부, 마르부
--------------------------------------------------
🔗 URL: https://www.youtube.com/@theforestofbeasts


일단 유튜브링크(중복제거)를 토대로 수집

In [30]:
import pandas as pd

# 1. 파일 불러오기 (한글 깨짐 대비)
try:
    df = pd.read_csv("bj_links_extracted_full.csv", encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv("bj_links_extracted_full.csv", encoding='cp949')

# 2. 네가 짠 코드 (단순히 채워져 있는 고유값 개수)
cnt_basic = df['youtube_url'].nunique()
print(f"단순히 채워진 고유값 개수: {cnt_basic}개")

# 3. [추천] 'youtube' 글자가 포함된 '진짜 유튜브 링크' 고유 개수
valid_urls = df['youtube_url'].dropna()
cnt_real = valid_urls[valid_urls.str.contains('youtube', na=False)].nunique()
print(f"진짜 유튜브 링크 고유 개수: {cnt_real}개")

# 두 개수 비교
if cnt_basic == cnt_real:
    print("오! 데이터가 아주 깔끔하네. 이상한 값이 하나도 없어!")
else:
    print(f"앗, {cnt_basic - cnt_real}개의 이상한 값(오타나 다른 링크)이 섞여 있었네!")

단순히 채워진 고유값 개수: 6118개
진짜 유튜브 링크 고유 개수: 6100개
앗, 18개의 이상한 값(오타나 다른 링크)이 섞여 있었네!


In [33]:
import yt_dlp
import pandas as pd
import time
import os

def get_youtube_subscribers(url):
    ydl_opts = {
        'quiet': True,
        'no_warnings': True,
        'extract_flat': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=False)
            channel_id = info.get('channel_id', '알 수 없음')
            channel_name = info.get('channel') or info.get('uploader', '알 수 없음')
            sub_count = info.get('channel_follower_count', None)
            
            return {
                '스트리머명(추출)': channel_name,
                '유튜브URL': url,
                '채널ID': channel_id,
                '구독자 수': sub_count
            }
        except Exception as e:
            print(f" [{url}] 에러 발생: {e}")
            return None

# 1. 파일 및 진행 상황 확인
target_file = "youtube_subscribers_result.csv"
done_urls = set()

if os.path.exists(target_file):
    try:
        df_existing = pd.read_csv(target_file, encoding='utf-8-sig')
        done_urls = set(df_existing['유튜브URL'].tolist())
        print(f" 기존 수집 기록 발견: {len(done_urls)}개는 건너뜁니다.")
    except Exception as e:
        print(f"   기존 파일을 읽는 중 오류 발생(새로 시작합니다): {e}")

# 2. 원본 데이터 로드
try:
    df_raw = pd.read_csv('bj_links_extracted_full.csv', encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv('bj_links_extracted_full.csv', encoding='cp949')

# 전체 타겟 URL 추출 (중복 제거)
all_urls = df_raw['youtube_url'].dropna()
all_urls = all_urls[all_urls.str.contains('youtube|youtu\.be', na=False)].drop_duplicates().tolist()

urls_to_crawl = [url for url in all_urls if url not in done_urls]

total_count = len(all_urls)
remaining_count = len(urls_to_crawl)

print(f" 전체 대상: {total_count}개 / 남은 대상: {remaining_count}개\n")

# 3. 크롤링 및 실시간 저장
print("수집 및 실시간 저장 시작")

for index, url in enumerate(urls_to_crawl, start=len(done_urls) + 1):
    print(f" [{index}/{total_count}] 수집 중... ({url})")
    
    result = get_youtube_subscribers(url)
    
    if result:
        # 결과 데이터프레임 생성
        df_new = pd.DataFrame([result])
        
        # 실시간 파일 저장 (mode='a'는 Append, 데이터가 있으면 헤더 제외)
        file_exists = os.path.isfile(target_file)
        df_new.to_csv(target_file, mode='a', index=False, 
                      header=not file_exists, encoding='utf-8-sig')
    
    time.sleep(1) # 매너 타임

print("\n" + "="*60)
print("모든 수집이 완료되었습니다!")
print("="*60)

<>:50: SyntaxWarning: invalid escape sequence '\.'
<>:50: SyntaxWarning: invalid escape sequence '\.'
C:\Users\anna3\AppData\Local\Temp\ipykernel_34408\3838521562.py:50: SyntaxWarning: invalid escape sequence '\.'
  all_urls = all_urls[all_urls.str.contains('youtube|youtu\.be', na=False)].drop_duplicates().tolist()


 기존 수집 기록 발견: 3개는 건너뜁니다.
 전체 대상: 6118개 / 남은 대상: 6115개

수집 및 실시간 저장 시작
 [4/6118] 수집 중... (https://www.youtube.com/@%EA%B9%80%EC%8A%A4%EC%95%BC%EC%A8%A9)


ERROR: [youtube:tab] @%EA%B9%80%EC%8A%A4%EC%95%BC%EC%A8%A9: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


 [https://www.youtube.com/@%EA%B9%80%EC%8A%A4%EC%95%BC%EC%A8%A9] 에러 발생: ERROR: [youtube:tab] @%EA%B9%80%EC%8A%A4%EC%95%BC%EC%A8%A9: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
 [5/6118] 수집 중... (https://www.youtube.com/@B-059-zj7dk)
 [6/6118] 수집 중... (https://www.youtube.com/@%EB%B4%84%EB%88%88-Bomnun_YouTube)
 [7/6118] 수집 중... (https://youtu.be/HPhupeC09Ec?si=4MGQwZxBe3C_ZLQ6)
 [8/6118] 수집 중... (https://www.youtube.com/channel/UCg80sqVlwbBx1tdJiTuzXBA)
 [9/6118] 수집 중... (https://www.youtube.com/channel/UCQwgofkUNm8Vj3rFw2reB2w)
 [10/6118] 수집 중... (https://www.youtube.com/@wkd1359)
 [11/6118] 수집 중... (https://www.youtube.com/@SR%EC%BB%B4%ED%8D%BC%EB%8B%88/videos)
 [12/6118] 수집 중... (https://www.youtube.com/@DarkHood-z6c/featured)
 [13/6118] 수집 중... (https://www.youtube.com/watch?v=GxldQ9eX2wo)
 [14/6118] 수집 중... (https://www.youtube.com/@dodo_YUKI)
 [15/6118] 수집 중... (https://www.youtube.com/@silver_waveKY/featured)
 [16/6118] 수집 중... (

ERROR: [youtube:tab] @0%ED%95%9C%EC%9A%B8: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


...

 [https://www.youtube.com/@%ED%98%84%EC%8B%9C%EA%B2%BD] 에러 발생: ERROR: [youtube:tab] @%ED%98%84%EC%8B%9C%EA%B2%BD: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
 [6079/6118] 수집 중... (https://www.youtube.com/@Yuchae1)
 [6080/6118] 수집 중... (https://www.youtube.com/@%ED%98%95%EC%9D%BC%EB%8F%99%EC%82%BC)
 [6081/6118] 수집 중... (https://www.youtube.com/channel/UCYtFUU_IAfZ9fSrbyaFwGzQ)
 [6082/6118] 수집 중... (https://www.youtube.com/channel/UCZBs6o2PcEX6SQc1RtUH1rA)
 [6083/6118] 수집 중... (https://www.youtube.com/@YouthTigerTube)
 [6084/6118] 수집 중... (https://www.youtube.com/@HOMA_Ch)
 [6085/6118] 수집 중... (https://www.youtube.com/@%ED%98%B8%EB%AF%B8%EC%98%A4-homioh)
 [6086/6118] 수집 중... (https://www.youtube.com/channel/UCXF01MmPdqnfIWDjywfRk8A)
 [6087/6118] 수집 중... (https://www.youtube.com/channel/UCYxzrym2ofD_uS-KYM9-MVQ)
 [6088/6118] 수집 중... (https://www.youtube.com/channel/UC5D9FY9lb_CUuLKt-icFkAw)
 [6089/6118] 수집 중... (https://www.youtube.com/@ha

ERROR: [youtube:tab] @0-FC_DragonGod-8: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


 [https://www.youtube.com/@0-FC_DragonGod-8] 에러 발생: ERROR: [youtube:tab] @0-FC_DragonGod-8: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
 [6091/6118] 수집 중... (https://www.youtube.com/@hood_VioletMoon)
 [6092/6118] 수집 중... (https://www.youtube.com/channel/UCOA6JlDdMf4_QIfjd_tmuEw)
 [6093/6118] 수집 중... (https://www.youtube.com/@hoochuny)
 [6094/6118] 수집 중... (https://www.youtube.com/channel/UC5U85Bd4xMq-_BeFnZ1eLjg)
 [6095/6118] 수집 중... (https://www.youtube.com/@Fukyo00)
 [6096/6118] 수집 중... (https://www.youtube.com/channel/UCGgi8I2ABCKnd3OVtiseAQg)
 [6097/6118] 수집 중... (https://www.youtube.com/@TechHunrang)
 [6098/6118] 수집 중... (https://www.youtube.com/channel/UC35FCiTyqJxeJaozHHe6C7Q)
 [6099/6118] 수집 중... (https://www.youtube.com/channel/UCnvzheBC0xt--XA2ZMTxQcw)
 [6100/6118] 수집 중... (https://www.youtube.com/@hwiheon_1)
 [6101/6118] 수집 중... (https://www.youtube.com/channel/UCgWFvH4lZEQGFWSGRwfNaRQ)
 [6102/6118] 수집 중... (https://www.youtu

ERROR: [youtube:tab] @DanArchive123: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)


 [https://www.youtube.com/@DanArchive123] 에러 발생: ERROR: [youtube:tab] @DanArchive123: Unable to download API page: HTTP Error 404: Not Found (caused by <HTTPError 404: Not Found>)
 [6116/6118] 수집 중... (https://www.youtube.com/@Arumi_Anima)
 [6117/6118] 수집 중... (https://www.youtube.com/@ppottattto)
 [6118/6118] 수집 중... (https://youtube.com/)

모든 수집이 완료되었습니다!


In [51]:
import pandas as pd

# 1. 파일 불러오기 (저장할 때 썼던 utf-8-sig 인코딩 사용)
try:
    df = pd.read_csv('youtube_subscribers_result.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("🚨 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 2. 전체 데이터 개수 파악
total_rows = len(df)

# 3. 항목별 결측치(빈칸 또는 알 수 없음) 개수 세기
# 스트리머명 점검
missing_names = df['스트리머명(추출)'].isna().sum()
unknown_names = (df['스트리머명(추출)'] == '알 수 없음').sum()

# 구독자 수 점검
missing_subs = df['구독자 수'].isna().sum()

# 4. 진단 리포트 출력
print("\n" + "="*50)
print("📊 유튜브 크롤링 데이터 건강 검진 리포트")
print("="*50)
print(f"✅ 전체 수집된 데이터 : {total_rows}개")
print("-" * 50)
print(f"👻 스트리머명이 아예 빈칸(null)인 경우  : {missing_names}개")
print(f"👻 스트리머명이 '알 수 없음'인 경우        : {unknown_names}개")
print(f"👻 구독자 수가 아예 빈칸(null)인 경우    : {missing_subs}개")
print("="*50)

# 5. 문제가 있는 데이터 눈으로 직접 확인하기 (최대 5개씩)
if missing_names > 0 or unknown_names > 0:
    print("\n[스트리머명에 문제가 있는 데이터 예시]")
    # 이름이 비어있거나 '알 수 없음'인 조건
    condition = df['스트리머명(추출)'].isna() | (df['스트리머명(추출)'] == '알 수 없음')
    print(df[condition][['유튜브URL', '채널ID']].head())

if missing_subs > 0:
    print("\n[구독자 수가 비어있는 데이터 예시]")
    # 구독자 수가 없는 채널의 이름과 주소 확인
    print(df[df['구독자 수'].isna()][['스트리머명(추출)', '유튜브URL', '채널ID']].head())


📊 유튜브 크롤링 데이터 건강 검진 리포트
✅ 전체 수집된 데이터 : 5933개
--------------------------------------------------
👻 스트리머명이 아예 빈칸(null)인 경우  : 0개
👻 스트리머명이 '알 수 없음'인 경우        : 0개
👻 구독자 수가 아예 빈칸(null)인 경우    : 124개

[구독자 수가 비어있는 데이터 예시]
    스트리머명(추출)                                             유튜브URL  \
4          봄눈  https://www.youtube.com/@%EB%B4%84%EB%88%88-Bo...   
81     고구 GO9                    https://www.youtube.com/@GO9_gg   
91        공구백                https://www.youtube.com/@gongguback   
179        뀨웅  https://www.youtube.com/channel/UCSR-ifIl_RgeF...   
236       뇌시경               https://www.youtube.com/@Noesigyeong   

                         채널ID  
4    UCacUmO-W3B9pQr_gnT0W9yA  
81   UCl2ndgppfhUJmWv9ninfVzA  
91                         43  
179                       126  
236                        10  


In [52]:
import pandas as pd

# 1. 수집 완료된 원본 파일 읽기
df = pd.read_csv('youtube_subscribers_result.csv', encoding='utf-8-sig')

# 2. 구독자 수가 비어있는 124개만 쏙 골라내기
missing_subs_df = df[df['구독자 수'].isna()]

# 3. 눈으로 확인하기 (채널ID에 숫자가 들어간 게 있는지 확인)
print("\n[구독자 수가 없는 데이터 10개 미리보기]")
print(missing_subs_df[['스트리머명(추출)', '채널ID', '유튜브URL']].head(10))

# 4. 124개만 따로 엑셀(CSV)로 저장해서 자세히 보기
missing_subs_df.to_csv('check_124_missing_subs.csv', index=False, encoding='utf-8-sig')
print("\n✅ 'check_124_missing_subs.csv' 파일이 생성되었습니다! 엑셀로 열어보세요.")



[구독자 수가 없는 데이터 10개 미리보기]
    스트리머명(추출)                      채널ID  \
4          봄눈  UCacUmO-W3B9pQr_gnT0W9yA   
81     고구 GO9  UCl2ndgppfhUJmWv9ninfVzA   
91        공구백                        43   
179        뀨웅                       126   
236       뇌시경                        10   
247        뉴잉                       424   
321      디오두목  UCqQ7wKMBWdthQY87Q4n_oZA   
376    루이 Ruy  UCLKDjzGpwnOGEMNbRki77Xw   
388        르화                        10   
435        매화                       156   

                                                유튜브URL  
4    https://www.youtube.com/@%EB%B4%84%EB%88%88-Bo...  
81                     https://www.youtube.com/@GO9_gg  
91                 https://www.youtube.com/@gongguback  
179  https://www.youtube.com/channel/UCSR-ifIl_RgeF...  
236               https://www.youtube.com/@Noesigyeong  
247              https://www.youtube.com/@newing_video  
321                  https://www.youtube.com/@dio00825  
376                  https://www.youtube.com

In [72]:
import pandas as pd

# 1. 파일 불러오기
try:
    df = pd.read_csv('youtube_subscribers_result.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("🚨 파일을 찾을 수 없습니다. 파일 이름과 위치를 확인해주세요.")
    exit()

# 2. 구독자 수가 비어있는(null) 행 개수 세기
missing_subs_count = df['구독자 수'].isna().sum()

# 3. 결과 출력
print("\n" + "="*50)
print(f"👻 구독자 수가 비어있는 행 개수: {missing_subs_count}개")
print("="*50)


👻 구독자 수가 비어있는 행 개수: 119개


In [73]:
import pandas as pd

# 1. 원본 파일 불러오기
try:
    df = pd.read_csv('youtube_subscribers_result.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("🚨 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 2. '구독자 수'가 비어있는(NaN) 행들만 싹 지우기
# subset=['구독자 수']를 지정하면 다른 칸은 비어있어도 넘어가고, 딱 구독자 수 칸이 빈 행만 지워.
df_cleaned = df.dropna(subset=['구독자 수'])

# 3. 새로운 파일로 안전하게 저장하기
# index=False를 해야 엑셀로 열었을 때 맨 앞에 쓸데없는 숫자(인덱스) 열이 안 생겨!
df_cleaned.to_csv('youtube_subscribers_cleaned.csv', index=False, encoding='utf-8-sig')

# 4. 결과 확인
print("\n" + "="*50)
print("🧹 데이터 청소 완료!")
print("="*50)
print(f"원래 데이터 개수 : {len(df)}개")
print(f"정리 후 데이터 개수: {len(df_cleaned)}개")
print(f"🗑️ 지워진 행 개수  : {len(df) - len(df_cleaned)}개")
print("\n✅ 'youtube_subscribers_cleaned.csv' 파일로 깔끔하게 저장되었습니다!")


🧹 데이터 청소 완료!
원래 데이터 개수 : 5932개
정리 후 데이터 개수: 5813개
🗑️ 지워진 행 개수  : 119개

✅ 'youtube_subscribers_cleaned.csv' 파일로 깔끔하게 저장되었습니다!


In [74]:
import pandas as pd

# 1. 청소된 파일 불러오기
try:
    df_cleaned = pd.read_csv('youtube_subscribers_cleaned.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("🚨 파일을 찾을 수 없습니다.")
    exit()

# 2. 결측치가 정말 0개인지 확인!
print("\n" + "="*50)
print("✨ 청소된 데이터 최종 건강 검진 ✨")
print("="*50)
print(f"✅ 살아남은 데이터 : {len(df_cleaned)}개")
print("-" * 50)
print(f"👻 스트리머명 빈칸 : {df_cleaned['스트리머명(추출)'].isna().sum()}개")
print(f"👻 채널ID 빈칸     : {df_cleaned['채널ID'].isna().sum()}개")
print(f"👻 구독자 수 빈칸  : {df_cleaned['구독자 수'].isna().sum()}개")
print("="*50)
print("전부 0개라면 완벽합니다! 바로 다음 작업으로 넘어가셔도 좋습니다 🚀")


✨ 청소된 데이터 최종 건강 검진 ✨
✅ 살아남은 데이터 : 5813개
--------------------------------------------------
👻 스트리머명 빈칸 : 0개
👻 채널ID 빈칸     : 0개
👻 구독자 수 빈칸  : 0개
전부 0개라면 완벽합니다! 바로 다음 작업으로 넘어가셔도 좋습니다 🚀


In [75]:
import pandas as pd

try:
    df_main = pd.read_csv('bj_links_extracted_full_with_x_followers_v2.csv', encoding='utf-8')
except UnicodeDecodeError:
    df_main = pd.read_csv('bj_links_extracted_full_with_x_followers_v2.csv', encoding='cp949')

df_subs = pd.read_csv('youtube_subscribers_cleaned.csv', encoding='utf-8-sig')

# 2. 데이터 병합 (Merge)
# main 데이터의 'youtube_url'과 subs 데이터의 '유튜브URL'이 같은 것끼리 짝지어줍니다. (Left Join)
df_merged = pd.merge(
    df_main, 
    df_subs[['유튜브URL', '구독자 수']], 
    left_on='youtube_url', 
    right_on='유튜브URL', 
    how='left'
)

# 3. 컬럼명 변경 및 불필요한 열 삭제
# '구독자 수'를 'youtube_subscribers'로 영문 변경
df_merged = df_merged.rename(columns={'구독자 수': 'youtube_subscribers'})
# 병합용으로 썼던 중복 열('유튜브URL')은 지워줍니다.
df_merged = df_merged.drop(columns=['유튜브URL'])

# 4. 컬럼 순서 재배치 (youtube_url 바로 옆에 붙이기)
cols = list(df_merged.columns)
# 맨 끝에 생성된 youtube_subscribers를 리스트에서 잠시 빼고
cols.remove('youtube_subscribers')
# youtube_url이 있는 위치를 찾아서 그 바로 다음 자리(+1)에 끼워 넣습니다.
youtube_url_idx = cols.index('youtube_url')
cols.insert(youtube_url_idx + 1, 'youtube_subscribers')

# 새로운 컬럼 순서로 데이터프레임 업데이트
df_final = df_merged[cols]

# 5. 최종 파일 저장
output_filename = 'bj_links_FINAL_completed.csv'
df_final.to_csv(output_filename, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print("데이터 병합 완료")
print("="*60)
print(f"새로 생성된 파일 이름: {output_filename}")
print("이제 엑셀에서 열어보시면 youtube_url 바로 옆에 youtube_subscribers가 예쁘게 들어있을 거예요! 🚀")


데이터 병합 완료
새로 생성된 파일 이름: bj_links_FINAL_completed.csv
이제 엑셀에서 열어보시면 youtube_url 바로 옆에 youtube_subscribers가 예쁘게 들어있을 거예요! 🚀
